In [1]:
from radpy.sedfit import *
from radpy.stellar import *

/home/oxfor/miniforge/envs/test_env/lib/python3.12/site-packages/radpy/sedfit.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Holoviews not imported. Some visualizations will not be available.
PyMultiNest not imported.  MultiNest fits will not work.
/home/oxfor/miniforge/envs/test_env/lib/python3.12/site-packages/astroARIADNE/fitter.py:43: UserWarning: (py)MultiNest installation (or libmultinest.dylib) not detected.
  warnings.warn(


Checking existing file to see if MD5 sum matches ...
File exists. Not overwriting.
Checking existing file to see if MD5 sum matches ...
File exists. Not overwriting.


In [2]:
starname = 'HD 16141'
star = StellarParams()

In [3]:
filename = "/mnt/c/Users/oxfor/Research/rsadpy/tests/test_data/photometry/HD16141mags.dat"
phot_data = read_in_photometry(filename)

In [4]:
phot_data

index,sed_filter,la,width,flux,eflux
,,Angstrom,Angstrom,erg / (Angstrom s cm2),erg / (Angstrom s cm2)
int64,str17,float64,float64,float64,float64
0,Johnson.U,3511.89,328.5,-7.9907398677168295,0.05723706750242215
1,TYCHO.TYCHO.B_MvB,4194.96,370.695,-7.617066603920657,0.022622874958050057
2,Johnson.B,4382.77,505.85,-7.528413331800076,0.0219343592986413
3,GAIA.GAIA3.Gbp,5109.71,1078.75,-7.480981151098504,0.02173094051801857
4,TYCHO.TYCHO.V_MvB,5300.19,566.775,-7.438174864233383,0.022140997133029652
5,Johnson.V,5501.4,444.9,-7.431643607322269,0.0219343592986413
6,GAIA.GAIA3.G,6217.59,2026.485,-7.477607877353822,0.021728844871176
7,GAIA.GAIA3.Grp,7769.02,1462.22,-7.476805293820858,0.02175309789060117


In [ ]:
star.logg = 4.14
star.logg_err = 0.207
star.feh = 0.14
star.feh_err = 0.01
ra_deg, dec_deg, ra_hms, dec_hms = pull_coords(starname, star, verbose = True)

In [ ]:
star.dist, star.dist_err = distances(starname, verbose = True)

In [ ]:
initial_guess = [5000,4.14, 0.14, 0.0]
sed_fit = fit_sed(phot_data, star, initial_guess, 'phoenix', teffrange = [4000,8000], fitT = True, verbose = False)

In [ ]:
def convert(sed, unit):
    ##########################################################
    # Function: convert                                      #
    # Inputs:                                                #
    #    sed: sed object                                     #
    #    unit: string with what unit you want                #
    #          options are: 'micron' or 'AA'                 #
    # Outputs:                                               #
    #    lit_w: input wavelength                             #
    #    lit_f: input flux                                   #
    #    lit_dw: input wavelength error                      #
    #    lit_df: input flux error                            #
    #    model_w: model wavelength                           #
    #    model_f: model flux                                 #
    #    synth_f: model fluxes in the wavelength badpasses   #
    # How it works:                                          #
    #    1. Reads in the model wavelength and converts to    #
    #       microns if needed. Default is angstroms          #
    #    2. Reads in the model flux and converts it into     #
    #       flux from log10(flux/wavelength) and converts it #
    #       into micron units if needed. Default is angstrom #
    #    3. Reads in wavelength error and converts if needed #
    #       Default is angstroms                             #
    #    4. Reads in flux error and converts it from log     #
    #       and converts into microns if needed.             #
    #    5. Reads in model wavelengths and converts it to    #
    #       microns if needed.                               #
    #    6. Reads in model flux and converts it out of log   #
    #       and converts it to microns if needed             #
    #    7. Reads in synthetic fluxes and converts it out of #
    #       log and converts it into microns if needed       #
    #    8. Returns values                                   #
    ##########################################################
    w = sed.sed['la']
    f = sed.sed['flux']
    dw = sed.sed['width']
    df = sed.sed['eflux']
    sf = sed.mags
    mw = sed.la
    mf = sed.fx.flatten()
    if unit == 'AA':
        lit_w = w  # literature wavelengths
        lit_f = (10 ** f) / np.array(w)  # literature flux
        lit_dw = dw # literature wavelength error
        lit_df = (df / 0.434) * lit_f  # literature flux error

        model_w = mw  # model wavelength
        model_f = (10 ** mf) / mw  # model flux

        synth_f = (10 ** sf) / np.array(w)  # model fluxes for the wavelength

        # residuals = lit_f-synth_f

        return np.array(lit_w), np.array(lit_f), np.array(lit_dw), np.array(lit_df), np.array(model_w), np.array(model_f), np.array(synth_f)

    if unit == 'micron':
        lit_w = w * (1e-4)  # literature wavelengths
        lit_f = ((10 ** f) / np.array(w)) * (1e4)  # literature flux
        lit_dw = dw * (1e-4)  # literature wavelength error
        lit_df = ((df / 0.434) * f) * (1e4)  # literature flux error

        model_w = np.array(sed.la) * (1e-4)  # model wavelength
        model_f = ((10 ** (sed.fx.flatten())) / model_w) * (1e4)  # model flux

        synth_f = ((10 ** sed.mags) / np.array(lit_w)) * (1e4)  # model fluxes for the wavelength

        # residuals = np.array(lit_f)-np.array(synth_f)

        return np.array(lit_w), np.array(lit_f), np.array(lit_dw), np.array(lit_df), np.array(model_w), np.array(
            model_f), np.array(synth_f)

In [ ]:
sed_fit.sed

In [ ]:
 litw, litf, litdw, litdf, model_w, model_f, synthf = convert(sed_fit, unit='AA')

In [ ]:
model_f

In [ ]:
f = sed_fit.sed['flux']
w = sed_fit.sed['la']
df = sed_fit.sed['eflux']


In [ ]:
def new_model(x):
    new_m = np.interp(x, model_w, model_f)
    return new_m

In [ ]:
result, error, info = quad(new_model, min(model_w), 1000000, full_output = True)

In [ ]:
print('Fbol = ', round(result/(1e-8), 5), '+/-', round(error/(1e-8),5), 'x10^(-8) erg/s/cm^s/angstrom')

In [ ]:
info

In [ ]:
fbol, fbol_err = calc_fbol(star, sed_fit, 'AA', verbose = True)

In [ ]:
(0.23049/5.00967)*100

In [ ]:
from SEDFit.sed import SEDFit
from radpy.sedfit import *

In [ ]:
def definefilter(self, tmass=True, cousins=True, gaia=True, galex=True, johnson=True,
                 panstarrs=True, sdss=True, wise=True, xmm=True, spitzer=True, tycho=True, hip=True, tess=True,
                 stromgren=True,
                 new=True, empty=False, **kwargs):
    idx = []
    if new:
        for i in range(len(self.sed)):
            self.sed['sed_filter'][i] = self.sed['sed_filter'][i].replace(':', '.').replace('/', '.')
            self.sed['width'] = 0 * u.AA
        if tmass:
            if new:
                self.sed['sed_filter'] = self.sed['sed_filter'].astype(object)
                for i in range(len(self.sed)):
                    # self.sed['sed_filter'][i]=self.sed['sed_filter'][i].replace('Johnson.J','2MASS.J')
                    # self.sed['sed_filter'][i]=self.sed['sed_filter'][i].replace('Johnson.H','2MASS.H')
                    # self.sed['sed_filter'][i]=self.sed['sed_filter'][i].replace('Johnson.K','2MASS.Ks')
                    self.sed['sed_filter'][i] = self.sed['sed_filter'][i].replace('UKIDSS.J', '2MASS.J')
                    self.sed['sed_filter'][i] = self.sed['sed_filter'][i].replace('UKIDSS.H', '2MASS.H')
                    self.sed['sed_filter'][i] = self.sed['sed_filter'][i].replace('UKIDSS.Ks', '2MASS.Ks')
                    self.sed['sed_filter'][i] = self.sed['sed_filter'][i].replace('UKIDSS.K', '2MASS.Ks')
                    self.sed['sed_filter'][i] = self.sed['sed_filter'][i].replace('VISTA.J', '2MASS.J')
                    self.sed['sed_filter'][i] = self.sed['sed_filter'][i].replace('VISTA.H', '2MASS.H')
                    self.sed['sed_filter'][i] = self.sed['sed_filter'][i].replace('VISTA.Ks', '2MASS.Ks')
                    self.sed['sed_filter'][i] = self.sed['sed_filter'][i].replace('VISTA.K', '2MASS.Ks')
                    if self.sed['sed_filter'][i] == '2MASS:K': self.sed['sed_filter'][i] = '2MASS.Ks'
                self.sed['sed_filter'] = self.sed['sed_filter'].astype(str)

            filters = ['2MASS.J', '2MASS.H', '2MASS.Ks']
            width = np.array([0.152026, 0.241018, 0.250619]) / 2 * u.micron
            la = np.array([12393.09, 16494.95, 21638.61]) * u.AA
            ind = np.array([39, 41, 43])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if cousins:
            filters = ['Cousins.U', 'Cousins.B', 'Cousins.V', 'Cousins.R', 'Cousins.I']
            width = np.array([0.0657, 0.10117, 0.55014, 0.13811, 0.101107]) / 2 * u.micron
            la = np.array([3511.89, 4382.77, 5501.4, 6414.42, 7858.32]) * u.AA
            ind = np.array([7, 12, 24, 27, 33])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if gaia:
            filters = ['GAIA.GAIA3.G', 'GAIA.GAIA3.Gbp', 'GAIA.GAIA3.Grp']
            width = np.array([0.405297, 0.21575, 0.292444]) / 2 * u.micron
            la = np.array([6217.59, 5109.71, 7769.02]) * u.AA
            ind = np.array([28, 18, 34])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if galex:
            filters = ['GALEX.FUV', 'GALEX.NUV']
            width = np.array([0.026557, 0.076831]) / 2 * u.micron
            la = np.array([1535.08, 2300.78]) * u.AA
            ind = np.array([0, 3])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if johnson:
            filters = ['Johnson.U', 'Johnson.B', 'Johnson.V', 'Johnson.R', 'Johnson.I', 'Johnson.J', 'Johnson.K',
                       'Johnson.H']
            width = np.array([0.0657, 0.10117, 0.08898, 0.207, 0.2316, 0.319355, 0.5785, 0.286263]) / 2 * u.micron
            la = np.array([3511.89, 4382.77, 5501.4, 6819.05, 8657.44, 12317.3, 21735.85, 16396.38]) * u.AA
            ind = np.array([9, 13, 23, 29, 36, 40, 44, 42])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if panstarrs:
            filters = ['PAN-STARRS.PS1.g', 'PAN-STARRS.PS1.r', 'PAN-STARRS.PS1.i', 'PAN-STARRS.PS1.z',
                       'PAN-STARRS.PS1.y']
            width = np.array([0.105308, 0.125241, 0.120662, 0.099772, 0.063898]) / 2 * u.micron
            la = np.array([4849.11, 6201.2, 7534.96, 8674.2, 9627.79]) * u.AA
            ind = np.array([16, 25, 30, 35, 38])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if sdss:
            filters = ['SDSS.u', 'SDSS.g', 'SDSS.r', 'SDSS.i', 'SDSS.z']
            width = np.array([0.054097, 0.106468, 0.105551, 0.110257, 0.116401]) / 2 * u.micron
            la = np.array([3556.52, 4702.5, 6175.58, 7489.98, 8946.71]) * u.AA
            ind = np.array([8, 17, 26, 31, 37])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if wise:
            filters = ['WISE.W1', 'WISE.W2', 'WISE.W3', 'WISE.W4']
            width = np.array([0.662642, 1.042266, 5.505523, 4.10168]) / 2 * u.micron
            la = np.array([33682.21, 46179.06, 120718.12, 221944.04]) * u.AA
            ind = np.array([45, 48, 51, 53])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if xmm:
            filters = ['XMM-OT.V', 'XMM-OT.B', 'XMM-OT.U', 'XMM-OT.UVW1', 'XMM-OT.UVM2', 'XMM-OT.UVW2']
            width = np.array([0.069956, 0.091023, 0.067513, 0.074398, 0.046194, 0.04355]) / 2 * u.micron
            la = np.array([5450.47, 4368.97, 3465.51, 2895.36, 2284.66, 2041.68]) * u.AA
            ind = np.array([21, 14, 6, 4, 2, 1])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if spitzer:
            filters = ['Spitzer.IRAC.3.6', 'Spitzer.IRAC.4.5', 'Spitzer.IRAC.5.8', 'Spitzer.IRAC.8.0',
                       'Spitzer.MIPS.24']
            width = np.array([0.683618, 0.864992, 1.256117, 2.52885, 5.296286]) / 2 * u.micron
            la = np.array([35378.41, 44780.49, 56961.78, 77978.40, 235937.78]) * u.AA
            ind = np.array([46, 47, 49, 50, 52])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if hip:
            filters = ['Hipparcos.Hipparcos.Hp_MvB']
            width = np.array([0.240569]) / 2 * u.micron
            la = np.array([5338.25]) * u.AA
            ind = np.array([20])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if tycho:
            filters = ['TYCHO.TYCHO.B_MvB', 'TYCHO.TYCHO.V_MvB']
            width = np.array([0.074139, 0.113355]) / 2 * u.micron
            la = np.array([4194.96, 5300.19]) * u.AA
            ind = np.array([11, 19])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if tess:
            filters = ['TESS.TESS.Red']
            width = np.array([0.389865]) / 2 * u.micron
            la = np.array([7697.6]) * u.AA
            ind = np.array([32])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))
        if stromgren:
            filters = ['Stromgren.u', 'Stromgren.v', 'Stromgren.b', 'Stromgren.y']
            width = np.array([0.037244, 0.022398, 0.020208, 0.02533]) / 2 * u.micron
            la = np.array([3443.6, 4105.46, 4666.25, 5475.17]) * u.AA
            ind = np.array([5, 10, 15, 22])
            idx.extend(self.selectflux(filters, width, la, ind, new=new, empty=empty))

        self.sed = self.sed[np.array(idx).flatten()]
        a = np.argsort(self.sed['la'])
        self.sed = self.sed[a]
        return


def downloadflux(self, userinput, deletevot=True, **kwargs):
    target = str(self.ra) + '%20' + str(self.dec)
    good = False
    if userinput is not None:
        self.sed = userinput
        # print("Using User input table")
        # print(self.sed)
        return
    else:
        if self.vizier_filename is not None:
            vot_fn = self.vizier_filename
            self.sed = Table.read(vot_fn)
            # print("Using VOT table")
            good = True
            if deletevot:
                os.remove(vot_fn)
        else:
            attempts, maxattempts = 0, 4
            while attempts < maxattempts:
                try:
                    target = str(self.ra) + '%20' + str(self.dec)
                    self.sed = Table.read(f"https://vizier.cds.unistra.fr/viz-bin/sed?-c={target}&-c.rs={self.radius}")
                    # print("Using vizier search")
                    good = True
                    break
                except:
                    attempts += 1
                    time.sleep(attempts ** 2)
        if not good:
            self.sed = []
            # print("Nothing worked")
            return

        self.sed['index'] = int(0)
        self.sed['la'] = self.sed['sed_freq'].to(u.AA, equivalencies=u.spectral())
        a = np.where((self.sed['la'] < 30 * u.micron) & (self.sed['la'] > 1000 * u.AA))[0]
        self.sed = self.sed[a]

        self.sed["sed_flux"] = self.sed["sed_flux"].to((u.erg / u.s / (u.cm ** 2) / u.AA),
                                                       equivalencies=u.spectral_density(self.sed['la'].data * u.AA))
        self.sed["sed_eflux"] = self.sed["sed_eflux"].to((u.erg / u.s / (u.cm ** 2) / u.AA),
                                                         equivalencies=u.spectral_density(self.sed['la'].data * u.AA))

        a = np.where(self.sed['sed_flux'] > 0)[0]
        self.sed = self.sed[a]

        a = np.where((np.isnan(self.sed["sed_eflux"]) == True) | (self.sed["sed_eflux"] / self.sed["sed_flux"] < 0.02))[
            0]
        self.sed["sed_eflux"][a] = self.sed["sed_flux"][a] * 0.02

        self.sed['eflux'] = self.sed["sed_eflux"] / self.sed["sed_flux"] / np.log(10)
        self.sed['flux'] = np.log10(self.sed["sed_flux"].value * self.sed['la'])

        self.definefilter(**kwargs)
        self.sed = self.sed[['index', 'sed_filter', 'la', 'width', 'flux', 'eflux']]
        # print("Using vizier photometry search")
        # print(self.sed)
        return


def set_quality(self):
    with open(pkg_resources.resource_filename('SEDFit', 'quality.p'), 'rb') as file:
        model = pickle.load(file)
    n = len(self.sed)
    input = np.zeros((n, 42, 2)) - 1
    input[:, self.sed['index'].astype(int), 0] = np.tile(np.max(self.sed['flux']) - self.sed['flux'], (n, 1))
    input[:, :, 1] = 0
    input[range(len(self.sed)), self.sed['index'], 1] = 1
    q = np.round(model.predict(input, verbose=0), 3)
    self.sed['valid'] = q[:, 3]
    self.sed['excess'] = q[:, 2]
    self.sed['bad'] = q[:, 1]
    a = np.where(q[:, 3] > 0.2)[0]
    if len(a) / n < 0.3:
        print(
            'Warning: large number of fluxes rejected, due to IR excess, noise, or misattribution. Manual vetting suggested')
        return False
    return

In [ ]:
x = SEDFit('2:35:19.92917', '-3:33:38.17294', 1, grid_type = 'phoenix')
downloadflux(x, phot_data)
set_quality(x)

In [ ]:
import numpy as np
def randomize_photometry(sed):
    idx = sed['index']
    filt = sed['sed_filter']
    wl = sed['la']
    width = sed['width']
    f = sed['flux']
    df = sed['eflux']

    #rand_wl = np.random.normal(wl, width)
    rand_f = np.random.normal(f, df)

    new_phot = pd.DataFrame(
        {'index': idx, 'sed_filter': filt, 'la': wl, 'width': width, 'flux': rand_f,
         'eflux': df})
    new_sed = Table.from_pandas(new_phot)
    new_sed['la'].unit = u.AA
    new_sed['width'].unit = u.AA
    new_sed['flux'] = rand_f
    new_sed['eflux'] = df
    new_sed['flux'].unit = u.erg / u.s / (u.cm ** 2) / u.AA
    new_sed['eflux'].unit = u.erg / u.s / (u.cm ** 2) / u.AA

    #if verbose:
    #    new_sed

    return new_sed

    
    

In [ ]:
def calc_fbol(star,x, unit, verbose = False):
    ##########################################################
    # Function: calc_fbol                                    #
    # Inputs:                                                #
    #    star: star object                                   #
    #    x: sed object                                       #
    #    unit: unit string                                   #
    # Outputs:                                               #
    #    result: bolometric flux                             #
    #    error: error on bolometric flux                     #
    # How it works:                                          #
    #    1. Calls convert to generate the model values       #
    #    2. Defines a "model" for integration purposes       #
    #    3. Integrates the model                             #
    #    4. Sets bolometric flux value and error to star     #
    #    5. Returns values.                                  #
    ##########################################################
    _, _, _, _, model_w, model_f, _ = convert(x, unit=unit)
    def new_model(x):
        new_m = np.interp(x, model_w, model_f)
        return new_m

    result, error = quad(new_model, min(model_w), 1000000)

    new_dfbol = np.sqrt((error**2)+(result*0.02)**2)
    
    if verbose:
        print('Fbol = ', round(result/(1e-8), 5), '+/-', round(new_dfbol/(1e-8),5), 'x10^(-8) erg/s/cm^2')

    star.fbol = round(result/(1e-8), 5)
    star.fbol_err = round(new_dfbol/(1e-8), 5)
    return result, new_dfbol

In [ ]:
initial_guess = [5000,4.14, 0.14, 0.0]
sed_fit = fit_sed(phot_data, star, initial_guess, 'phoenix', teffrange = [4000,8000], fitT = True, verbose = True)

In [ ]:
fb, dfb = calc_fbol(star, sed_fit, 'AA', verbose = True)

In [ ]:
def fit_func(sed, star, num_iter, initial_guess, model, teffrange=None, loggrange=None, fehrange=None, avrange = None, 
             fitT=False, fit_logg=False,fit_feh=False, fit_av = False, verbose=False):
    d = star.dist
    ra = star.ra_hms
    dec = star.dec_dms

    f = io.StringIO()
    with contextlib.redirect_stdout(f):
        x = SEDFit(ra, dec, 1, grid_type=model)

    x.dist = d
    teff, logg, feh, av = initial_guess

    x.addguesses(teff=teff, logg=logg, feh=feh, av = av)
    if teffrange is not None:
        x.addrange(teff=teffrange)
    if loggrange is not None:
        x.addrange(logg=loggrange)
    if fehrange is not None:
        x.addrange(feh=fehrange)
    if avrange is not None:
        x.addrange(av=avrange)

    fbol = []
    fbol_err = []
    #mcrad = []
    #mcteff = []
    #mclogg = []
    #mcfeh = []
    #mcav = []
    chi2s = []
    chi2reds = []
    seds = []
    
    for i in range(num_iter):
        print("Starting fit #", i)
        rand_sed = randomize_photometry(sed)
        downloadflux(x, rand_sed)
        set_quality(x)
        x.fit(use_gaia=False, idx=np.arange(0, len(x.sed['index'])), fitdist=False, fitteff=fitT, fitfeh=fit_feh, 
              fitlogg=fit_logg, fitav = fit_av, quality_check=False)
        
        #mcrad.append(x.getr()[0])
        numOparams = 1
        if fitT == True:
            numOparams += 1
            # print("fitT is set to True.")
            # print(numOparams)
            #star.sed_teff = x.getteff()[0]
            #mcteff.append(x.getteff())
        if fit_logg == True:
            numOparams += 1
            # print("fitlogg is set to True.")
            # print(numOparams)
            #mclogg.append(x.getlogg())
            #star.sed_logg = x.getlogg()[0]
        if fit_feh == True:
            numOparams += 1
            # print("fitfeh is set to True.")
            # print(numOparams)
            #mcfeh.append(x.getfeh())
            #star.sed_feh = x.getfeh()
        if fit_av == True:
            numOparams +=1
            #star.sed_av = x.getav()
            #mcav.append(x.getav())

        chi2, chi2r = chi2red(x, numOparams, verbose=False)
        chi2s.append(chi2)
        chi2reds.append(chi2r)

        fb, dfb = calculate_fbol(star, x, unit, verbose = False)
        fbol.append(fb)
        fbol_error.append(dfb)

        seds.append(x)

        print("Finished fit #", i)

    min_chi2r = min(chi2r)
    minidx = np.argmin(chi2r)

    min_sed = seds[minidx]
    min_chi = chi2[minidx]
    min_fbol = fbol[minidx]
    min_dfbol = fbol_error[minidx]
    
    if fitT == True:
        star.sed_teff = min_sed.getteff()[0]
    if fit_logg == True:
        star.sed_logg = min_sed.getlogg()[0]
    if fit_feh == True:
        star.sed_feh = min_sed.getfeh()
    if fit_av == True:
        star.sed_av = min_sed.getav()

    new_fbolerr = np.sqrt((min_dfbol**2)+(min_fbol*0.02)**2)
    if verbose:
        print("Distance: {} pc".format(x.getdist()))
        print("AV: {} mag".format(x.getav()))
        print("Radius: {} Rsun".format(x.getr()))
        print("Teff: {} K".format(x.getteff()))
        print("Log g: {} ".format(x.getlogg()))
        print("Fe/H: {}".format(x.getfeh()))
        print("Fbol: ", round(min_fbol/1e-8, 5), "+/-", round(new_fbolerr/1e-8, 5), "erg/s/cm^s")
        
    return x, min_fbol/1e-8, new_fbolerr/1e-8

In [ ]:
teff = 5000
logg = 4.14
feh = 0.14
av = 0
model = 'phoenix'
sed = phot_data
num_iter = 5
fitT = True
fit_feh = False
fit_logg = False
fit_av = False
unit = 'AA'

In [ ]:
d = star.dist
ra = star.ra_hms
dec = star.dec_dms

f = io.StringIO()
with contextlib.redirect_stdout(f):
    x = SEDFit(ra, dec, 1,use_gaia_params = False, use_gaia_xp = False, grid_type=model)

x.dist = d
#teff, logg, feh, av = initial_guess
x.addguesses(teff=teff, logg=logg, feh=feh, av = av)
x.addrange(teff = [4000,8000])

fbol = []
fbol_err = []
chi2s = []
chi2reds = []
seds = []

for i in range(num_iter):
    sed_obj = x
    print("Starting fit #", i+1)
    #rand_sed = randomize_photometry(sed)
    #downloadflux(sed_obj, rand_sed)
    downloadflux(sed_obj, sed)
    set_quality(sed_obj)
    sed_obj.fit(use_gaia=False, idx=np.arange(0, len(sed_obj.sed['index'])), fitdist=False, fitteff=fitT, fitfeh=fit_feh, 
                fitlogg=fit_logg, fitav = fit_av, quality_check=False)
    numOparams = 1
    if fitT == True:
        numOparams += 1
    if fit_logg == True:
        numOparams += 1 
    if fit_feh == True:
        numOparams += 1
    if fit_av == True:
        numOparams +=1

    chi2, chi2r = chi2red(sed_obj, numOparams, verbose=True)
    chi2s.append(chi2)
    chi2reds.append(chi2r)

    fb, dfb = calc_fbol(star, sed_obj, unit, verbose = True)
    fbol.append(fb)
    fbol_err.append(dfb)

    seds.append(sed_obj)

    print("Finished fit #", i+1)

In [ ]:
min_chi2r = min(chi2reds)
minidx = np.argmin(chi2reds)

min_sed = seds[minidx]
min_chi = chi2s[minidx]
min_fbol = fbol[minidx]
min_dfbol = fbol_err[minidx]
    
if fitT == True:
    star.sed_teff = min_sed.getteff()[0]
if fit_logg == True:
    star.sed_logg = min_sed.getlogg()[0]
if fit_feh == True:
    star.sed_feh = min_sed.getfeh()
if fit_av == True:
    star.sed_av = min_sed.getav()

new_fbolerr = np.sqrt((min_dfbol**2)+(min_fbol*0.02)**2)
#if verbose:
print("Distance: {} pc".format(x.getdist()))
print("AV: {} mag".format(x.getav()))
print("Radius: {} Rsun".format(x.getr()[0]))
print("Teff: {} K".format(x.getteff()[0]))
print("Log g: {} ".format(x.getlogg()[0]))
print("Fe/H: {}".format(x.getfeh()))
print("Fbol: ", round(min_fbol/1e-8, 5), "+/-", round(new_fbolerr/1e-8, 5), "erg/s/cm^s")
print("Precision: ", round((new_fbolerr/min_fbol)*100, 3), "%")

In [ ]:
sed_obj.sed

In [ ]:
phot_data

In [ ]:
def fit_func(sed, star, num_iter, initial_guess, model, teffrange=None, loggrange=None, fehrange=None, avrange = None, 
             fitT=False, fit_logg=False,fit_feh=False, fit_av = False, verbose=False):

In [ ]:
initial_guess = [5000,4.14, 0.14, 0.0]
fit_func(phot_data, star, 2, initial_guess, 'phoenix', teffrange = [4000,8000], fitT = True, verbose = True)

In [ ]:
initial_guess = [5000,4.14, 0.14, 0.0]
sed_fit = fit_sed(phot_data, star, initial_guess, 'phoenix', teffrange = [4000,8000], fitT = True, verbose = False)

In [ ]:
((8.167411991014701e-09)/(2.1631036405934208e-07))*100

In [ ]:
tables = []
rand_sed1 = randomize_photometry(phot_data)
rand_sed2 = randomize_photometry(phot_data)

In [ ]:
x1 = SEDFit('2:35:19.92917', '-3:33:38.17294', 1, grid_type = 'phoenix')
downloadflux(x1, rand_sed1)
set_quality(x1)

In [ ]:
x2 = SEDFit('2:35:19.92917', '-3:33:38.17294', 1, grid_type = 'phoenix')
downloadflux(x2, rand_sed1)
set_quality(x2)

In [ ]:
test = []
test.append(x2)

In [ ]:
test.append(x1)

In [ ]:
test[0].sed

In [ ]:
1/np.log(10)

In [ ]:
import dustmaps.sfd
dustmaps.sfd.fetch()

In [ ]:
starname = 'HD 1469'
star = StellarParams()
ra_deg, dec_deg, ra_hms, dec_hms = pull_coords(starname, star, verbose = True)
gaia_id = pull_gaia_id(starname, star, verbose = True)

In [ ]:
filename = '/mnt/c/Users/oxfor/Research/HD158259/Data/test_photometry.csv'
phot_data = read_in_photometry(filename)

In [ ]:
logg = 4.21
dlogg = 0.33
m = 0.0
dm = 0.12

star.logg = logg
star.logg_err = dlogg
star.feh = m
star.feh_err = dm

In [ ]:
D, dD = distances('HD 222155', verbose = True)
#star.dist = D
#star.dist_err = dD

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Disable GPU
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  #suppress everything but error messages
import tensorflow as tf

# Print devices to check if GPU is being used
print("GPU devices available: ", tf.config.list_physical_devices("GPU"))

In [ ]:
sed_fit = fit_sed(phot_data, star,  [5850, 4.21, 0.0, 0.0],  model = 'phoenix', teffrange = [5000,7000], fitT = True, verbose = True)

In [ ]:
sed_fit.getr()

In [ ]:
def setaxislabels(exp, unit, logplot = False, fbol_lam = False):
    ##########################################################
    # Function: setaxislabels                                #
    # Inputs:                                                #
    #    unit: string of unit wanted                         #
    #    fbol_lam: flag for fbol_lam                         #
    # Outputs:                                               #
    #    xlab: x axis label                                  #
    #    ylab: y axis label                                  #
    # How it works:                                          #
    #    1. Based on unit chosen, sets x label dependent on  #
    #       unit                                             #
    #    2. Based on unit chosen and if the fbol_lam flag    #
    #       has been set, sets y label                       #
    #    3. Returns x axis and y axis labels                 #
    ##########################################################
    if logplot:
        if fbol_lam:
            ylab = r'$\rm \lambda F_{\lambda}~[\frac{erg}{cm^{2}~s}$]'
        else:
            if unit == 'AA':
                ylab = r'$\rm F_{\lambda}~[\frac{erg}{cm^{2}~s~\AA}$]'
            elif unit == 'micron':
                ylab = r'$\rm F_{\lambda}~[\frac{erg}{cm^{2}~s~\mu m}$]'
        if unit == 'AA':
            xlab = r'$\rm Wavelength~[\AA]$'
        elif unit == 'micron':
            xlab = r'$\rm Wavelength~[\mu m]$'
        return xlab, ylab
    else:
        if fbol_lam:
            #print('Fbol lam')
            #print('Exponent:', exp)
            if exp < 0:
                #print('Exp < 0')
                ylab = rf'$\rm \lambda F_{{\lambda}}~[\times 10^{{{exp}}}~\frac{{\rm erg}}{{\rm cm^2~s}}]$'
            if unit == 'AA':
                #print('angstroms')
                xlab = r'$\rm Wavelength~[\AA]$'
            elif unit == 'micron':
                #print('Microns')
                xlab = r'$\rm Wavelength~[\mu m]$'
            return xlab, ylab
        else:
            #print('No fbol lam')
            #print('Exoponent:', exp)
            if unit == 'AA':
                #print('Angstroms')
                xlab = r'$\rm Wavelength~[\AA]$'
                if exp < 0:
                    #print('exp < 0')
                    ylab = rf'$\rm F_{{\lambda}}~[\times 10^{{{exp}}}~\frac{{\rm erg}}{{\rm cm^2~s~\AA}}]$'
            elif unit == 'micron':
                #print('Microns')
                xlab = r'$\rm Wavelength~[\mu m]$'
                if exp < 0:
                    #print('Exp < 0')
                    ylab = rf'$\rm F_{{\lambda}}~[\times 10^{{{exp}}}~\frac{{\rm erg}}{{\rm cm^2~s~\mu m}}]$'
            return xlab, ylab

In [ ]:
def set_values(x, unit, logplot=False, fbol_lam=False, verbose=False):
    ##########################################################
    # Function: set_values                                   #
    # Inputs:                                                #
    #    x: sed object                                       #
    #    unit: unit string                                   #
    #    logPlot: default is False, if True, indicates the   #
    #             log flag                                   #
    #    fbol_lam: default is False, if True, indicates the  #
    #             fbol_lam is flag                           #
    # Outputs:                                               #
    #    litp_xvals: input wavelength                        #
    #    litp_yvals: input flux                              #
    #    litp_dxvals: input wavelength error                 #
    #    litp_dyvals: input flux error                       #
    #    model_xvals: model wavelength                       #
    #    model_yvals: model flux                             #
    #    synth_yvals: model fluxes in the wavelength         #
    #                 bandpasses                             #
    #   residuals: lit flux values minus synth values        #
    # How it works:                                          #
    #    1. Calls convert to generate the model values       #
    #    2. Based on the flags set, converts the values to   #
    #       match what the flags say                         #
    #       logplot: convert everything back into log10      #
    #       fbol_lam: multiply the flux by wavelength        #
    #    3. Returns values                                   #
    ##########################################################
    iwave, iflux, idwave, idflux, mw, mf, msf = convert(x, unit=unit)

    if logplot and fbol_lam:
        # print('set values: Log and lambda')
        model_xvals = np.log10(mw)
        model_yvals = np.log10(mf * mw)
        litp_xvals = np.log10(iwave)
        litp_yvals = np.log10(iwave * iflux)
        litp_dyvals = 0.434 * (idflux / iflux)
        litp_dxvals = 0.434 * (idwave / iwave)
        synth_yvals = np.log10(msf * iwave)
        res = litp_yvals - synth_yvals
        exp = 0
        
        return model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp

    if logplot and not fbol_lam:
        # print('set values Log and no lambda')
        model_xvals = np.log10(mw)
        model_yvals = np.log10(mf)
        litp_xvals = np.log10(iwave)
        litp_yvals = np.log10(iflux)
        litp_dyvals = 0.434 * (idflux / iflux)
        litp_dxvals = 0.434 * (idwave / iwave)
        synth_yvals = np.log10(msf)
        res = litp_yvals - synth_yvals
        exp = 0
        return model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp

    if not logplot and not fbol_lam:
        # print('set values No log and no lambda')
        number = iflux[0]
        _, exp = normalize_number(number)
        # print('Dividing by:', exp)
        model_xvals = mw
        model_yvals = mf / (10 ** exp)
        litp_xvals = iwave
        litp_yvals = iflux / (10 ** exp)
        litp_dyvals = idflux / (10 ** exp)
        litp_dxvals = idwave
        synth_yvals = msf / (10 ** exp)
        res = litp_yvals - synth_yvals

        return model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp

    if not logplot and fbol_lam:
        number = (iflux[0] * iwave[0])
        _, exp = normalize_number(number)
        # print('Dividing by:', exp)
        model_xvals = mw
        model_yvals = (mf * mw) / (10 ** exp)
        litp_xvals = iwave
        litp_yvals = (iflux * iwave) / (10 ** exp)
        litp_dyvals = (idflux) / (10 ** exp)
        litp_dxvals = idwave
        synth_yvals = (msf * iwave) / (10 ** exp)
        res = litp_yvals - synth_yvals

        return model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp

In [ ]:
def convert_fit_params(star_name, star, fit_params_dict, verbose = False):
    fits = fit_params_dict.get(star_name, {})
    
    fitT = fits['fitTeff']
    fitLG = fits['fitLogg']
    fitFEH = fits['fitFeh']
    fitAV = fits['fitAv']
        
    init_logg = star.logg
    init_feh = star.feh
    init_av = 0
    if fitT:
        teff_range = ast.literal_eval(fits['Trange'])
        init_teff = (teff_range[0] + teff_range[1]) / 2
    else:
        init_teff = 5000
        teff_range = None
    if fitLG:
        logg_range = ast.literal_eval(fits['Loggrange'])
    else:
        logg_range = None
    if fitFEH:
        feh_range = ast.literal_eval(fits['Fehrange'])
    else:
        feh_range = None
    if fitAV:
        av_range = ast.literal_eval(fits['Avrange'])
    else:
        av_range = None

    init_vals = [init_teff, init_logg, init_feh, init_av]
    ranges = [teff_range, logg_range, feh_range, av_range]
    fitflags = [fitT, fitLG, fitFEH, fitAV]
    return init_vals, ranges, fitflags

In [ ]:
def set_res_axis(res, logplot = False):
    min_res = np.min(res)
    max_res = np.max(res)

    minres = round(min_res, 1)
    maxres = round(max_res, 1)

    if abs(minres) >= abs(maxres):
        if logplot:
            res_axis_min = (abs(minres)+0.05)*-1
            res_axis_max = abs(minres)+0.05
            res_loc = [abs(minres)*-1, 0, abs(minres)]
            res_labels = [rf'$\rm {val}$' for val in res_loc]
        else:
            res_axis_min = (abs(minres)+0.5)*-1
            res_axis_max = abs(minres)+0.5
            res_loc = [round(abs(minres)*-1), 0, round(abs(minres))]
            res_labels = [rf'$\rm {val}$' for val in res_loc]
    elif abs(minres) <= abs(maxres):
        if logplot:
            res_axis_min = (abs(maxres)+0.05)*-1
            res_axis_max = abs(maxres)+0.05
            res_loc = [abs(maxres)*-1, 0, abs(maxres)]
            res_labels = [rf'$\rm {val}$' for val in res_loc]
        else:
            res_axis_min = (abs(maxres)+0.5)*-1
            res_axis_max = abs(maxres)+0.5
            res_loc = [round(abs(maxres)*-1), 0, round(abs(maxres))]
            res_labels = [rf'$\rm {val}$' for val in res_loc]

    return res_axis_min, res_axis_max, res_loc, res_labels

In [ ]:
def setaxisticklabels(iwave, iflux, exp, unit, set_axis, logplot=False, fbol_lam=False, verbose=False):
    print('setaxisticklabels')
    if set_axis:
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
        if logplot:
            # y-ticks
            y_loc = [round(ymin), round(ymax)]
            y_labels = [rf'$10^{{{y_loc[0]}}}$', rf'$10^{{{y_loc[1]}}}$']

            # x-ticks
            xaxis = np.linspace(xmin, xmax, 5)
            x_loc = []
            xl = []
            for i in range(len(xaxis)):
                xloc = (10 ** (xaxis[i]))

                if unit == 'AA':
                    xl.append(int((round(xloc, -3))))
                    x_loc.append(np.log10((round(xloc, -3))))
                if unit == 'micron':
                    if xloc < 0.8:
                        xl.append(round(xloc, 1))
                        x_loc.append(np.log10(round(xloc, 1))) 
                    else:
                        xl.append(round(xloc))
                        x_loc.append(np.log10(round(xloc)))
            x_labels = [rf'${(val)}$' for val in xl]
            return x_loc, x_labels, y_loc, y_labels
        else:
            if fbol_lam:
                # y ticks
                y_loc = np.linspace(0, ymax / (10 ** exp), 4)
                y_labels = [rf'$\rm {round(val)}$' for val in y_loc]

                # x ticks
                xaxis = np.linspace(xmin, xmax, 5)
                x_loc = []
                xl = []
                for i in range(len(xaxis)):
                    xloc = xaxis[i]
                    if unit == 'AA':
                        xl.append(int((round(xloc, -3))))
                        x_loc.append((round(xloc, -3)))
                    if unit == 'micron':
                        if xloc < 0.8 and xloc > 0:
                            xl.append(round(xloc,2))
                            x_loc.append(round(xloc,2))
                        elif xloc == 0:
                            xl.append(round(xloc))
                            x_loc.append(round(xloc))
                        else:
                            xl.append(np.floor(xloc))
                            x_loc.append(np.floor(xloc))
                            
                x_labels = [rf'${(val)}$' for val in xl]
                return x_loc, x_labels, y_loc, y_labels
            else:
                # y ticks
                y_loc = np.linspace(0, ymax / (10 ** exp), 4)
                y_labels = [rf'$\rm {round(val)}$' for val in y_loc]

                # x ticks
                xaxis = np.linspace(xmin, xmax, 5)
                x_loc = []
                xl = []
                for i in range(len(xaxis)):
                    xloc = xaxis[i]
                    if unit == 'AA':
                        xl.append(int((round(xloc, -3))))
                        x_loc.append((round(xloc, -3)))
                    if unit == 'micron':
                        if xloc < 0.8 and xloc > 0:
                            xl.append(round(xloc,2))
                            x_loc.append(round(xloc,2))
                        elif xloc == 0:
                            xl.append(round(xloc))
                            x_loc.append(round(xloc))
                        else:
                            xl.append(np.floor(xloc))
                            x_loc.append(np.floor(xloc))
                            
                x_labels = [rf'${(val)}$' for val in xl]
                return x_loc, x_labels, y_loc, y_labels

    elif set_axis is None:
        set_axis = set_radpy_axis_limits(iwave, iflux, exp, unit, logplot = logplot)
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
        #print('RADPy definied axis limits:', set_axis)
        #axis is already in np.log10 units
        if logplot:
            #print('Log plot')
            xmin = round((xmin), 1)
            xmax = round((xmax), 1)
            ymin = round(ymin)
            ymax = round(ymax)
            #print('Xmin:', xmin)
            #print('Xmax:', xmax)
            #print('Ymin:', ymin)
            #print('Ymax:', ymax)
            # y ticks
            y_loc = [ymin, ymax]
            y_labels = [rf'$10^{{{y_loc[0]}}}$', rf'$10^{{{y_loc[1]}}}$']
            
            # x ticks
            xaxis = np.linspace(xmin, xmax, 5)
            #xaxis is in np.log10 space
            #print('xaxis:', xaxis)
            x_loc = []
            xl = []
            for i in range(len(xaxis)):
                #converting out of log10 space
                xloc = (10 ** (xaxis[i]))
                #print(f'xloc = 10^{xaxis[i]} -> {xloc}')
                if unit == 'AA':
                    #print('Angstroms')
                    #xlabel is out of log10 space
                    xl.append(int((round(xloc, -3))))
                    #xlocation is in log10 space
                    x_loc.append(np.log10((round(xloc, -3))))
                if unit == 'micron':
                    #print('micron')
                    if xloc < 0.8:
                        #print('xloc < 0.8')
                        #xlabel is out of log10 space
                        xl.append(round(xloc, 1))
                        #xlocation is in log10 space
                        x_loc.append(np.log10(round(xloc, 1)))
                        #print('new x_loc:', x_loc)
                    else:
                        #print('xloc > 0.8')
                        #xlabel is out of log10 space
                        xl.append(round(xloc))
                        #xlocation is in log10 space
                        x_loc.append(np.log10(round(xloc)))
                        #print('new x_loc:', x_loc)

            x_labels = [rf'$\rm {(val)}$' for val in xl]
            #print('X location:', x_loc)
            #print('X labels:', x_labels)
            #print('Y location:', y_loc)
            #print('Y labels:', y_labels)
            return x_loc, x_labels, y_loc, y_labels
        else:
            #print('No log')
            #y axis limits are in 10^exp space
            if fbol_lam:
                #print('Fbol lam')
                # y ticks
                #taking the yvals out of 10^exp space
                yloc = np.linspace(ymin / (10 ** exp), ymax / (10 ** exp), 4)
                y_loc = [round(val) for val in yloc]
                y_labels = [rf'$\rm {round(val)}$' for val in y_loc]

                # x ticks
                xaxis = np.linspace(xmin, xmax, 5)
                x_loc = []
                xl = []
                for i in range(len(xaxis)):
                    xloc = xaxis[i]
                    #print('xloc:', xloc)
                    if unit == 'AA':
                        #print('Angstroms')
                        xl.append(int((round(xloc, -3))))
                        x_loc.append((round(xloc, -3)))
                    if unit == 'micron':
                        #print('Microns')
                        if xloc < 1 and xloc > 0:
                            #print('xloc < 1 and xloc > 0')
                            xl.append(round(xloc,2))
                            x_loc.append(round(xloc,2))
                            #print('new x_loc:', x_loc)
                        elif xloc == 0:
                            #print('xloc = 0')
                            xl.append(round(xloc))
                            x_loc.append(round(xloc))
                            #print('new x_loc:', x_loc)
                        else:
                            #print('xloc > 1')
                            #print("xloc mod1:", xloc%1)
                            if xloc%1 < 0.5:
                                #print("xlocmod1 < 0.5")
                                xl.append(round(np.floor(xloc)))
                                x_loc.append(round(np.floor(xloc)))
                            elif xloc%1 > 0.5:
                                #print("xlocmod1 > 0.5")
                                xl.append(round((xloc)))
                                x_loc.append(round((xloc)))
                            
                x_labels = [rf'${(val)}$' for val in xl]
                #print('X location:', x_loc)
                #print('X labels:', x_labels)
                #print('Y location:', y_loc)
                #print('Y labels:', y_labels)
                return x_loc, x_labels, y_loc, y_labels
            else:
                #print('No fbol_lam')
                # y ticks
                yloc = np.linspace(1, ymax / (10 ** exp), 4)
                y_loc = [round(val) for val in yloc]
                y_labels = [rf'$ \rm {round(val)}$' for val in y_loc]
                print('setaxisticklabels')
                print('ymin:', ymin / (10**exp))
                print('y_loc:', y_loc)
                # x ticks
                xaxis = np.linspace(xmin, xmax, 5)
                x_loc = []
                xl = []
                for i in range(len(xaxis)):
                    xloc = xaxis[i]
                    #print('xloc:', xloc)
                    if unit == 'AA':
                        #print('angstroms')
                        xl.append(int((round(xloc, -3))))
                        x_loc.append((round(xloc, -3)))
                    if unit == 'micron':
                        #print('microns')
                        if xloc < 1 and xloc > 0:
                            #print('xloc < 1 and xloc > 0')
                            xl.append(round(xloc,2))
                            x_loc.append(round(xloc,2))
                            #print('new x_loc:', x_loc)
                        elif xloc == 0:
                            #print('xloc = 0')
                            xl.append(round(xloc))
                            x_loc.append(round(xloc))
                            #print('new x_loc:', x_loc)
                        else:
                            #print('xloc > 1')
                            #print("xloc mod1:", xloc%1)
                            if xloc%1 < 0.5:
                                #print("xlocmod1 < 0.5")
                                xl.append(round(np.floor(xloc)))
                                x_loc.append(round(np.floor(xloc)))
                            elif xloc%1 > 0.5:
                                #print("xlocmod1 > 0.5")
                                xl.append(round((xloc)))
                                x_loc.append(round((xloc)))
                                
                            #print('new x_loc:', x_loc)
                            
                x_labels = [rf'${(val)}$' for val in xl]
                #print('X location:', x_loc)
                #print('X labels:', x_labels)
                #print('Y location:', y_loc)
                #print('Y labels:', y_labels)
                return x_loc, x_labels, y_loc, y_labels

In [ ]:
def setaxisticklabels(iwave, iflux, exp, unit, set_axis, logplot=False, fbol_lam=False, verbose=False):
    if set_axis:
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
    elif set_axis is None:
        set_axis = set_radpy_axis_limits(iwave, iflux, exp, unit, logplot = logplot)
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
        
    if logplot:
        xmin = round((xmin), 1)
        xmax = round((xmax), 1)
        ymin = round(ymin)
        ymax = round(ymax)
        # y ticks
        y_loc = [ymin, ymax]
        y_labels = [rf'$10^{{{y_loc[0]}}}$', rf'$10^{{{y_loc[1]}}}$']
            
        # x ticks
        xaxis = np.linspace(xmin, xmax, 5)
        #xaxis is in np.log10 space
        x_loc = []
        xl = []
        for i in range(len(xaxis)):
            #converting out of log10 space
            xloc = (10 ** (xaxis[i]))
            if unit == 'AA':
                #xlabel is out of log10 space
                xl.append(int((round(xloc, -3))))
                #xlocation is in log10 space
                x_loc.append(np.log10((round(xloc, -3))))
            if unit == 'micron':
                if xloc < 0.8:
                    #xlabel is out of log10 space
                    xl.append(round(xloc, 1))
                    #xlocation is in log10 space
                    x_loc.append(np.log10(round(xloc, 1)))
                else:
                    #xlabel is out of log10 space
                    xl.append(round(xloc))
                    #xlocation is in log10 space
                    x_loc.append(np.log10(round(xloc)))

        x_labels = [rf'$\rm {(val)}$' for val in xl]
        return x_loc, x_labels, y_loc, y_labels
    else:
        #y axis limits are in 10^exp space
        if fbol_lam:
            # y ticks
            #taking the yvals out of 10^exp space
            yloc = np.linspace(ymin / (10 ** exp), ymax / (10 ** exp), 4)
            y_loc = [round(val) for val in yloc]
            y_labels = [rf'$\rm {round(val)}$' for val in y_loc]

            # x ticks
            xaxis = np.linspace(xmin, xmax, 5)
            x_loc = []
            xl = []
            for i in range(len(xaxis)):
                xloc = xaxis[i]
                if unit == 'AA':
                    xl.append(int((round(xloc, -3))))
                    x_loc.append((round(xloc, -3)))
                if unit == 'micron':
                    if xloc < 1 and xloc > 0:
                        xl.append(round(xloc,2))
                        x_loc.append(round(xloc,2))
                    elif xloc == 0:
                        xl.append(round(xloc))
                        x_loc.append(round(xloc))
                    else:
                        if xloc%1 < 0.5:
                            xl.append(round(np.floor(xloc)))
                            x_loc.append(round(np.floor(xloc)))
                        elif xloc%1 > 0.5:
                            xl.append(round((xloc)))
                            x_loc.append(round((xloc)))
                            
            x_labels = [rf'${(val)}$' for val in xl]
            return x_loc, x_labels, y_loc, y_labels
        else:
            # y ticks
            yloc = np.linspace(1, ymax / (10 ** exp), 4)
            y_loc = [round(val) for val in yloc]
            y_labels = [rf'$ \rm {round(val)}$' for val in y_loc]
            # x ticks
            xaxis = np.linspace(xmin, xmax, 5)
            x_loc = []
            xl = []
            for i in range(len(xaxis)):
                xloc = xaxis[i]
                if unit == 'AA':
                    xl.append(int((round(xloc, -3))))
                    x_loc.append((round(xloc, -3)))
                if unit == 'micron':
                    if xloc < 1 and xloc > 0:
                        xl.append(round(xloc,2))
                        x_loc.append(round(xloc,2))
                    elif xloc == 0:
                        xl.append(round(xloc))
                        x_loc.append(round(xloc))
                    else:
                        if xloc%1 < 0.5:
                            xl.append(round(np.floor(xloc)))
                            x_loc.append(round(np.floor(xloc)))
                        elif xloc%1 > 0.5:
                            xl.append(round((xloc)))
                            x_loc.append(round((xloc)))
                            
            x_labels = [rf'${(val)}$' for val in xl]
            return x_loc, x_labels, y_loc, y_labels

In [ ]:
def set_radpy_axis_limits(w, f, exp, unit, logplot):
    ##########################################################
    # Function: set_axis_labels                              #
    # Inputs:                                                #
    #    mw: model wavelength array                          #
    #    mf: model flux array                                #
    #    unit: string of unit wanted                         #
    #    fbol_lam: flag for fbol_lam                         #
    # Outputs:                                               #
    #    set_axis: the axis limits in the format of          #
    #              [xmin, xmax, ymin, ymax]                  #
    # How it works:                                          #
    #    1. Based on unit chosen and fbol_lam flag setting,  #
    #       sets the axis limits based on the minimum of the #
    #       arrays  and maxs of the arrays                   #
    #    2. Returns the sxis limits                          #
    ##########################################################
    xmin = min(w)
    xmax = max(w)
    ymin = min(f)
    ymax = max(f)

    
    if unit == 'AA':
        if logplot:
            Xmin = xmin-(xmin*0.05)
            Xmax = xmax+(xmax*0.05)
            Ymin = ymin+(ymin*0.05)
            Ymax = ymax-(ymax*0.05)

            new_axis = [round(Xmin, 1),round(Xmax, 1), round(Ymin)*(10**(exp)), round(Ymax)*(10**(exp))]

            return new_axis
        else:
            Xmin = xmin-(xmin*0.1)
            Xmax = xmax+(xmax*0.1)
            Ymin = ymin+(ymin*0.1)
            Ymax = ymax+(ymax*0.2)

            new_axis = [round(Xmin, -2), round(Xmax,-3), round(Ymin, 1)*(10**(exp)), round(Ymax)*(10**(exp))]

            return new_axis
    if unit == 'micron':
        if logplot:
            Xmin = xmin-(xmin*0.05)
            Xmax = xmax+(xmax*0.05)
            Ymin = ymin+(ymin*0.01)
            Ymax = ymax-(ymax*0.01)
            
            new_axis = [round(Xmin, 2),round(Xmax, 2), round(Ymin)*(10**(exp)), round(Ymax)*(10**(exp))]

            return new_axis
        else:
            Xmin = xmin-(xmin*0.05)
            Xmax = xmax+(xmax*0.05)
            Ymin = ymin-(ymin*0.05)
            Ymax = ymax+(ymax*0.1)
            
            new_axis = [round(Xmin, 2), round(Xmax,2), round(Ymin, 1)*(10**(exp)), round(Ymax,1)*(10**(exp))]

            return new_axis

In [ ]:
def setaxislimits(iwave, iflux, exp, unit, set_axis, logplot=False, fbol_lam=False):
    print('setaxislimits:')
    if set_axis:
        #print('User definied axis limits:', set_axis)
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
        if logplot:
            #print('Log plot')
            #print('Xlimit: ', xmin - 0.1, xmax + 0.1)
            #print('ylimit: ', ymin - 0.1, ymax + 0.1)
            return xmin - 0.1, xmax + 0.1, ymin - 0.1, ymax + 0.1
        else:
            #print('no log plot')
            #if fbol_lam:
                #print('Flam')
            #else:
                #print('No flam')
            if unit == 'AA':
                #print('X limits:', xmin - 500, xmax + 500)
                #print('Y limits:', (ymin / (10 ** (exp))) - 0.25, (ymax / (10 ** (exp))) + 0.25)
                return xmin - 500, xmax + 500, (ymin / (10 ** (exp))) + 0.25, (ymax / (10 ** (exp))) - 0.25
            if unit == 'micron':
                #print('X limits:', xmin - 0.1, xmax + 0.1)
                #print('Y limits: ', (ymin / (10 ** (exp))) - 0.25, (ymax / (10 ** (exp))) + 0.25)
                return xmin - 0.1, xmax + 0.1, (ymin / (10 ** (exp))) - 0.25, (ymax / (10 ** (exp))) + 0.25

    else:
        set_axis = set_radpy_axis_limits(iwave, iflux, exp, unit, logplot = logplot)
        print('RADPy definied axis limits:', set_axis)
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]

        if logplot:
            #output of set_radpy_axis_limits are in log10 space
            #print('log plot')
            xmin = round((xmin), 1)
            xmax = round((xmax), 1)
            ymin = round((ymin), 1)
            ymax = round((ymax), 1)
            #print('X limits:', xmin - 0.1, xmax + 0.1)
            #print('Y limits:', ymin + 0.2, ymax + 0.1)
            return xmin - 0.1, xmax + 0.1, ymin - 0.5, ymax + 0.1

        else:
            #output of set_radpy_axis_limits are in yvals * 10^exp
            #print('No log plot')
            #if fbol_lam:
                #print('Flam')
            #else:
                #print('No flam')
            if unit == 'AA':
                print('X limits:', xmin - 500, xmax + 500)
                print('Y limits: ', (ymin / (10 ** (exp))) + 1, (ymax / (10 ** (exp))) + 0.25)
                return xmin - 500, xmax + 500, ((ymin) / (10 ** (exp)))-0.25, (ymax / (10 ** (exp))) + 0.25
            if unit == 'micron':
                #print('X limits:', xmin - 0.1, xmax + 0.1)
                #print('Y limits: ', (ymin / (10 ** (exp))) - 0.25, (ymax / (10 ** (exp))) + 0.25)
                return xmin - 0.1, xmax + 0.1, (ymin / (10 ** (exp))) -0.25, (ymax / (10 ** (exp))) + 0.25

In [ ]:
def setaxislimits(iwave, iflux, exp, unit, set_axis, logplot=False, fbol_lam=False):
    if set_axis:
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
    else:
        set_axis = set_radpy_axis_limits(iwave, iflux, exp, unit, logplot = logplot)
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
        
    if logplot:
        return xmin - 0.1, xmax + 0.1, ymin - 0.5, ymax + 0.1
    else:
        if unit =='AA':
            return xmin - 500, xmax + 500, (ymin / (10 ** (exp))) - 0.25, (ymax / (10 ** (exp))) + 0.25
        if unit == 'micron':
            return xmin - 0.1, xmax + 0.1, (ymin / (10 ** (exp))) - 0.25, (ymax / (10 ** (exp))) + 0.25


In [ ]:
def plot_sed(x, unit, logplot=True, fbol_lam=True, set_axis=None, title=None, savefig=None, uselatex = False, show=True, verbose=False):
    ##########################################################
    # Function: plot_sed                                     #
    # Inputs:                                                #
    #    x: sed object                                       #
    #    unit: unit chosen                                   #
    #    logplot: log flag                                   #
    #             if True, sets plot in log space            #
    #    fbol_lam: fbol_lam flag                             #
    #             if True, multiplies the flux by wavelength #
    #    set_axis: allows user to set their axis limits      #
    #             if None, will set based on the model vals  #
    #    title: allows user to set the plot title            #
    #    savefig: allows user to save fig                    #
    #            give a filename                             #
    #    show: shows Figure                                  #
    # Outputs:                                               #
    #    displays the plot                                   #
    # How it works:                                          #
    #    1. Determines the axis limits based on user input   #
    #       and flags set                                    #
    #    2. Calls set_values to generate the data to be      #
    #       plotted                                          #
    #    3. Plots everything                                 #
    ##########################################################

    #iwave, iflux, idwave, idflux, mw, mf, msf = convert(x, unit)

    plt.rcParams.update({'font.size': 15})
    plt.rcParams['xtick.direction'] = 'in'
    plt.rcParams['ytick.direction'] = 'in'
    plt.rcParams['text.usetex'] = uselatex

    f, axes = plt.subplots(2, 1, gridspec_kw={'height_ratios': [10, 3]}, sharex=True)

    model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp = set_values(x, unit,
                                                                                                                   logplot=logplot,
                                                                                                                   fbol_lam=fbol_lam,
                                                                                                                   verbose=verbose)
    xmin, xmax, ymin, ymax = setaxislimits(litp_xvals, litp_yvals, exp, unit, set_axis, logplot=logplot, fbol_lam=fbol_lam)
    xloc, xlabels, yloc, ylabels = setaxisticklabels(litp_xvals, litp_yvals, exp, unit, set_axis, logplot=logplot, fbol_lam=fbol_lam)

    axes[0].set_ylim(ymin, ymax)
    axes[1].set_xlim(xmin, xmax)
    axes[0].set_yticks(yloc)
    axes[0].set_yticklabels(ylabels)
    axes[1].set_xticks(xloc)
    axes[1].set_xticklabels(xlabels)

    axes[0].plot(model_xvals, model_yvals, 'g', linewidth=1, label=r'$\rm Model~Spectrum$')
    axes[0].plot(litp_xvals, litp_yvals, 'b.', markersize=10, markerfacecolor='none', label=r'$\rm Photometry$')
    axes[0].errorbar(litp_xvals, litp_yvals, xerr=litp_dxvals, yerr=litp_dyvals, fmt='.', markerfacecolor='none',
                     color='blue', capsize=3)
    axes[0].plot(litp_xvals, synth_yvals, 'r.', markersize=10, label=r'$\rm Synthetic~Photometry $')

    axes[0].legend(prop={'size': 10}, loc='best')
    axes[0].tick_params(axis='x', labelbottom=False)

    axes[1].plot(litp_xvals, res, 'k.')
    axes[1].errorbar(litp_xvals, res, yerr=litp_dyvals, fmt='.', color='black')
    axes[1].axhline(y=0)
    ramin, ramax, rloc, rlabel = set_res_axis(res, logplot = logplot)
    axes[1].set_ylim(ramin, ramax)
    axes[1].set_yticks(rloc)
    axes[1].set_yticklabels(rlabel)
    if logplot:
        if unit == 'micron':
            axes[1].set_ylabel(r'$\rm Residuals$', labelpad=5)
        if unit == 'AA':
            axes[1].set_ylabel(r'$\rm Residuals$', labelpad=5)
    else:
        if unit == 'micron':
            axes[1].set_ylabel(r'$\rm Residuals$', labelpad=0)
        if unit == 'AA':
            axes[1].set_ylabel(r'$\rm Residuals$', labelpad=0)

    xlab, ylab = setaxislabels(exp, unit, logplot=logplot, fbol_lam=fbol_lam)
    axes[1].set_xlabel(xlab)
    axes[0].set_ylabel(ylab)
    plt.subplots_adjust(wspace=0, hspace=0)
    axes[0].xaxis.set_minor_locator(AutoMinorLocator())
    axes[0].yaxis.set_minor_locator(AutoMinorLocator())
    axes[1].xaxis.set_minor_locator(AutoMinorLocator())
    axes[1].yaxis.set_minor_locator(AutoMinorLocator())

    if title:
        axes[0].set_title(title)
    if savefig:
        f.savefig(savefig, bbox_inches='tight')
    if show:
        plt.show()

    return f, axes

In [ ]:
#set_axis = [-0.5, 0.5, -12, -11]      # log plot, flam, micron
#set_axis = [-0.5, 0.5, -12, -10.8]    # log plot, no flam, micron
set_axis = [0, 3, 0e-12, 7e-12]       # no log plot, flam, micron
#set_axis = [0.0, 3, 0e-12, 12e-12 ]     # no log plt, no flam, micron

#set_axis = [3.5, 4.5, -8, -7]         # log plot, flam, AA
#set_axis = [3.5, 4.5, -13, -10.7]     # log plot, no flam, AA
#set_axis = [0, 30000, 0e-8, 7e-8]     # no log plot, flam, AA
#set_axis = [0, 30000, 0e-12, 11e-12]  # no log plot, no flam, AA

#set_axis = None

plot_sed(sed_fit, 'micron', logplot = False, fbol_lam = True, set_axis = set_axis, title = None, savefig = None, show = True)